In [8]:
import os
import time
import warnings
from dotenv import load_dotenv
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# 1. Force the usage limit warning to raise an Exception
warnings.filterwarnings("error", message=".*usage limit.*", category=UserWarning)

# 2. Load environment variables
env_path = os.path.abspath(os.path.join(os.getcwd(), '..', '.env'))
load_dotenv(dotenv_path=env_path)

token = os.getenv('IBM_QUANTUM_TOKEN')
instance_env = os.getenv('IBM_QUANTUM_INSTANCE')
instance = instance_env if instance_env else None

if not token:
    raise ValueError("IBM_QUANTUM_TOKEN not found. Please verify your .env file path and contents.")

# 3. Authenticate
print("Connecting to IBM Quantum...")
service = QiskitRuntimeService(
    channel="ibm_quantum_platform", 
    token=token, 
    instance=instance
)

# 4. Select backend
print("Searching for the least busy real quantum backend...")
backend = service.least_busy(operational=True, simulator=False)
print(f"Selected backend: {backend.name}")

# 5. Prepare circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
sampler = Sampler(mode=backend)

# 6. Loop configuration
TARGET_MINUTES = 15
target_seconds = TARGET_MINUTES * 60
accumulated_seconds = 0
iteration = 1

print(f"\nStarting execution loop. Target: {TARGET_MINUTES} minutes ({target_seconds} seconds) of QPU time.")

# 7. Execution loop
while accumulated_seconds < target_seconds:
    print(f"\n--- Iteration {iteration} ---")
    try:
        # Submitting the job
        # If the usage limit is hit here, it will now raise an exception
        job = sampler.run([isa_circuit], shots=10000)
        print(f"Job ID: {job.job_id()} - Submitted. Waiting for results...")
        
        # Wait for completion
        result = job.result()
        
        # Wait for IBM's telemetry servers to sync the usage metrics
        print("Result received! Waiting 5 seconds for metrics to sync...")
        time.sleep(5) 
        
        # Extract metrics
        metrics = job.metrics()
        qpu_time = metrics.get('usage', {}).get('quantum_seconds', 0)
        
        # Fallback to usage_estimation if metrics are still 0
        if qpu_time == 0 and hasattr(job, 'usage_estimation'):
            estimation = job.usage_estimation
            if isinstance(estimation, dict):
                qpu_time = estimation.get('quantum_seconds', 0)
        
        accumulated_seconds += qpu_time
        
        print(f"Job consumed: {qpu_time:.2f} seconds.")
        print(f"Total accumulated in this session: {accumulated_seconds:.2f} / {target_seconds} seconds ({(accumulated_seconds/60):.2f} / {TARGET_MINUTES} minutes).")
        
        iteration += 1
        
    except Exception as e:
        error_msg = str(e).lower()
        # This will now successfully catch the converted warning
        if "quota" in error_msg or "limit" in error_msg or "usage limit" in error_msg or "429" in error_msg or "403" in error_msg:
            print("\nMonthly free tier exhausted (Quota Exceeded)! Stopping the loop safely.")
            break
        else:
            print(f"\nAn unexpected error occurred: {e}")
            break

print("\n--- Script finished ---")
if accumulated_seconds >= target_seconds:
    print(f"Success! Reached the target of {TARGET_MINUTES} minutes.")


qiskit_runtime_service._discover_account:WARNING:2026-08-24 19:02:04,733: Loading account with the given token. A saved account will not be used.


Connecting to IBM Quantum...
Searching for the least busy real quantum backend...
Selected backend: ibm_fez

Starting execution loop. Target: 15 minutes (900 seconds) of QPU time.

--- Iteration 1 ---

Monthly free tier exhausted (Quota Exceeded)! Stopping the loop safely.

--- Script finished ---
